# H28 E0 Technical Control

**Historical preparation — superseded for the real E0 flow on 2026-10-06.**

The operator used **Submit to Competition → File Upload** and uploaded the frozen `submission.zip` directly. The real run completed **Succeeded / score 0.05**. This notebook materialization route was unnecessary. Retain it as historical preparation; it is not evidence that a notebook executed for E0 and does not authorize another submission. See `harness_cert/reports/H28_KAGGLE_E0_CERTIFICATION_2026-10-06.md`. All notebook code below is preserved unchanged.

Frozen Git HEAD: `3c6133ea9a1d9440183bfacc72b9f6e823dc6208`

Expected ZIP SHA256: `25d07b6484d3c4fb4683170ed85b7a3adfe2ff0c1962f7949e43bc3cf87e4fc3`

Expected ZIP size: `443572` bytes.

Purpose: technical submission-path validation only. No optimization.

Attach a private input dataset containing only `submission.zip.bytes`: an unchanged copy of the frozen ZIP under a transport filename. Kaggle documents automatic extraction of uploaded archives; this filename is an experiment design to retain the bytes, not a guarantee about upload processing. Require the literal input file and matching hash below. The dataset slug and mount prefix are assigned by Kaggle.

Use CPU and Internet OFF. Run all cells in order. This notebook uses only the Python standard library to copy and verify one file. It imports no agent, reads no task content, runs no inference, and creates no predictions. Verify competition association and actual settings in the live UI.


In [ ]:
from pathlib import Path
import hashlib
import shutil
import zipfile

EXPECTED_HEAD = "3c6133ea9a1d9440183bfacc72b9f6e823dc6208"
EXPECTED_SHA256 = "25d07b6484d3c4fb4683170ed85b7a3adfe2ff0c1962f7949e43bc3cf87e4fc3"
EXPECTED_SIZE = 443572
INPUT_NAME = "submission.zip.bytes"
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_PATH = Path("/kaggle/working/submission.zip")
STAGING_PATH = OUTPUT_PATH.with_name(".h28-e0-submission.tmp")
EXPECTED_MEMBERS = (
    "adapters/main_lora/adapter_config.json",
    "adapters/main_lora/adapter_model.safetensors",
    "adapters/tool_lora/adapter_config.json",
    "adapters/tool_lora/adapter_model.safetensors",
    "agent.yaml",
    "configs/sampling.yaml",
    "eval_config.yaml",
    "prompts/analyzer.md",
    "prompts/system.md",
    "sub_agents/code_analyzer.yaml",
)

# These are the only two output files this notebook owns. Clear earlier runs.
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.unlink(missing_ok=True)
STAGING_PATH.unlink(missing_ok=True)


def verify_identity(path):
    size = path.stat().st_size
    if size != EXPECTED_SIZE:
        raise ValueError(f"E0-INPUT-MATERIALIZATION: size {size} != {EXPECTED_SIZE}")
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != EXPECTED_SHA256:
        raise ValueError(f"E0-INPUT-MATERIALIZATION: SHA256 {digest} != {EXPECTED_SHA256}")
    return size, digest


def verify_archive(path):
    with zipfile.ZipFile(path, "r") as archive:
        names = archive.namelist()
        if len(names) != len(EXPECTED_MEMBERS) or set(names) != set(EXPECTED_MEMBERS):
            raise ValueError("E0-PACKAGE: expected exactly the ten frozen archive members")
        if "agent.yaml" not in names or any(info.is_dir() for info in archive.infolist()):
            raise ValueError("E0-PACKAGE: expected regular files and archive-root agent.yaml")
        bad_member = archive.testzip()
        if bad_member is not None:
            raise ValueError(f"E0-PACKAGE: failed ZIP CRC check: {bad_member}")
    return names


In [ ]:
# Search directory names for one exact basename; never read other input files.
if not INPUT_ROOT.is_dir():
    raise FileNotFoundError("E0-INPUT-MATERIALIZATION: /kaggle/input is unavailable")
candidates = sorted(path for path in INPUT_ROOT.rglob(INPUT_NAME) if path.is_file())
print(f"input_candidates={len(candidates)}")
if len(candidates) != 1:
    raise ValueError("E0-INPUT-MATERIALIZATION: attach exactly one submission.zip.bytes file")
if candidates[0].is_symlink():
    raise ValueError("E0-INPUT-MATERIALIZATION: input must be a regular file, not a symlink")
source_path = candidates[0].resolve()
if not source_path.is_relative_to(INPUT_ROOT.resolve()):
    raise ValueError("E0-INPUT-MATERIALIZATION: source must remain inside /kaggle/input")
print(f"source_path={source_path}")


In [ ]:
source_size, source_sha256 = verify_identity(source_path)
verify_archive(source_path)
print(f"source_size={source_size}")
print(f"source_sha256={source_sha256}")
print("source_zip_integrity=PASS")


In [ ]:
# Copy file content unchanged. Verify the staged copy before publishing the ZIP.
try:
    shutil.copyfile(source_path, STAGING_PATH)
    verify_identity(STAGING_PATH)
    verify_archive(STAGING_PATH)
    STAGING_PATH.replace(OUTPUT_PATH)
except BaseException:
    STAGING_PATH.unlink(missing_ok=True)
    OUTPUT_PATH.unlink(missing_ok=True)
    raise


In [ ]:
try:
    output_size, output_sha256 = verify_identity(OUTPUT_PATH)
    members = verify_archive(OUTPUT_PATH)
except BaseException:
    OUTPUT_PATH.unlink(missing_ok=True)
    raise

print("output_zip_integrity=PASS")
print("zip_members:")
for member in members:
    print(f"  {member}")
print("H28_E0_ARTIFACT_READY")
print(f"path={OUTPUT_PATH}")
print(f"size={output_size}")
print(f"sha256={output_sha256}")


## Historical operator checkpoint (2026-10-05)

`H28_E0_ARTIFACT_READY` verifies the output bytes only. At preparation time H28 remained `NOT-REPRODUCED` until real Kaggle evidence was reviewed. The separate 2026-10-06 certification now records the actual direct-upload result; these retained instructions apply only to the historical proposal.

Create a completed saved notebook version and verify its output contains `submission.zip` with size `443572` and the exact printed hash. The likely save controls are **Save Version / Save & Run All — LIVE UI CONFIRMATION REQUIRED**.

**STOP before submitting.** Inspect and record the live competition/version/output controls. If an output selector is offered, the intended artifact is `submission.zip`. Do not substitute `submission.parquet`; the evaluation infrastructure creates task predictions later. Submit only through the operator's separate manual step in the runbook. This notebook contains no submission API.
